# Exploration/dev notebook for scheduling

In [1]:
import os
import pandas as pd
from pathlib import Path
import sqlalchemy as sa
import sqlalchemy.orm as so
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

In [2]:
REGISTRY_PATH = ROOT / "scoop" / "configuration" / "profiles" / "nsclc" / "nsclc_protocol_registry.yaml"

In [3]:
HEMONC_ENGINE_URL = os.environ["HEMONC_ENGINE_URL"]
def _require_engine_url(engine_url: str | None = None) -> str:
    resolved = engine_url or HEMONC_ENGINE_URL
    if not resolved:
        raise ValueError("Set HEMONC_ENGINE_URL or pass engine_url explicitly.")
    return resolved
engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))

In [4]:
from scoop.configuration.protocol_registry import RegistryYaml
from scoop.utils.scheduling import SystemicPhase
from scoop.utils.treatment_containers import ProtocolGenerator

In [5]:
with so.Session(engine) as session:
    nsclc_registry = RegistryYaml.from_yaml_path(REGISTRY_PATH, session)

In [6]:
nsclc_trt_generator = ProtocolGenerator(source_phases=nsclc_registry.source_phases, phase_unions=nsclc_registry.union_phases)

In [7]:
nsclc_registry.protocols

{'port_adjuvant_osimertinib': ProtocolYaml(protocol_id='port_adjuvant_osimertinib', source_phase_ids=['surgery_only', 'port', 'adj_osimertinib'], optional_phases=None),
 'surgery_then_port': ProtocolYaml(protocol_id='surgery_then_port', source_phase_ids=['surgery_only', 'port'], optional_phases=None),
 'neoadjuvant_or_perioperative_ici_w_chemo': ProtocolYaml(protocol_id='neoadjuvant_or_perioperative_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}}),
 'adjuvant_cisplatin_osimertinib': ProtocolYaml(protocol_id='adjuvant_cisplatin_osimertinib', source_phase_ids=['adjuvant_cisplatin_based', 'adjuvant_osimertinib'], optional_phases=None),
 'conc_crt_followed_by_durvalumab': ProtocolYaml(protocol_id='conc_crt_followed_by_durvalumab', source_phase_ids=['conc_crt', 'consolidation_durvalumab'], optional_phases=None),
 'conc_crt_foll

## Getting dates for a systemic phase
(The durvalumab sig bundle has implicit portions, each w/ dif cycle length and is rolled out incorrectly if done with each sig independent from table)

In [8]:
with so.Session(engine) as session:
    neoadj_or_peri = nsclc_trt_generator.get_protocols_from_protocol_yaml(
        nsclc_registry.protocols["neoadjuvant_or_perioperative_ici_w_chemo"], session
        )

In [9]:
neoadj_w_durvalumab = neoadj_or_peri[1].phases[0].options[0]
neoadj_w_durvalumab

Cisplatin and Docetaxel (DC) and Durvalumab Variant #01 (CUI: 130688)
Cisplatin: 100 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4936
Docetaxel: 85 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4937
Durvalumab: 750 mg IV once on day(s) 1 in cycle(s) 4,5. Neoadjuvant SigID: 4938

In [10]:
neoadj_durv_schedule = SystemicPhase.from_component_sigs(neoadj_w_durvalumab.sigs)
neoadj_durv_schedule

SystemicPhase(cycle_sets=[CycleSet(seed=None, cycle_length_lb=21, cycle_length_ub=21, cycle_len_unit='day', cycle_sigs='21-day cycle for 3 cycles', trios=[Trio(per_cycle_days=PerCycleDays(every=[1], base=[1], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='Cisplatin', cui=105, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b5fc270>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=100.0, dosemaxnum=100.0, doseunit='mg/m^2', doseunit_cui=134448, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='primary systemic', class_field='iv intermittent canonical sig'), cycle_events=CycleEvents(seed=None, every=[1, 2, 3], base=[1, 2, 3], optional=[], ongoing_step=None, first_event=1), other_info=OtherSigInfo(in_parens=None, tail='-', temp=None, sig_id=4936, date_added=datetime.datet

In [11]:
neoadj_durv_trt_dates = neoadj_durv_schedule.get_scheduled_dates(pd.to_datetime("2015-01-01"))
neoadj_durv_trt_dates

defaultdict(list,
            {Timestamp('2015-01-02 00:00:00'): [ComponentPart(component=Component(name='Cisplatin', cui=105, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b5fc270>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=100.0, dosemaxnum=100.0, doseunit='mg/m^2', doseunit_cui=134448, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='primary systemic', class_field='iv intermittent canonical sig'),
              ComponentPart(component=Component(name='Docetaxel', cui=164, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b618050>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=85.0, dosemaxnum=85.0, doseunit='mg/m^2', doseunit_cui=134448, divided=False, dose_cap=DoseCap(dosecapnum=No

In [12]:
for date, component_parts in neoadj_durv_trt_dates.items():
    dosing = {component_part.component.name: component_part.get_dose()
              for component_part in component_parts}
    print(f"{date}: {dosing}")

2015-01-02 00:00:00: {'Cisplatin': (100.0, 'mg/m^2'), 'Docetaxel': (85.0, 'mg/m^2')}
2015-01-23 00:00:00: {'Cisplatin': (100.0, 'mg/m^2'), 'Docetaxel': (85.0, 'mg/m^2')}
2015-02-13 00:00:00: {'Cisplatin': (100.0, 'mg/m^2'), 'Docetaxel': (85.0, 'mg/m^2')}
2015-03-06 00:00:00: {'Durvalumab': (750.0, 'mg')}
2015-03-20 00:00:00: {'Durvalumab': (750.0, 'mg')}


In [13]:
for date, component_parts in neoadj_durv_trt_dates.items():
    dosing = {component_part.component.name: component_part.get_dose(units_as_cuis=True)
              for component_part in component_parts}
    print(f"{date}: {dosing}")

2015-01-02 00:00:00: {'Cisplatin': (100.0, 134448), 'Docetaxel': (85.0, 134448)}
2015-01-23 00:00:00: {'Cisplatin': (100.0, 134448), 'Docetaxel': (85.0, 134448)}
2015-02-13 00:00:00: {'Cisplatin': (100.0, 134448), 'Docetaxel': (85.0, 134448)}
2015-03-06 00:00:00: {'Durvalumab': (750.0, 134444)}
2015-03-20 00:00:00: {'Durvalumab': (750.0, 134444)}


In [14]:
neoadj_w_nivolumab_eg = neoadj_or_peri[1].phases[0].options[2]
neoadj_w_nivolumab_eg

Cisplatin and Docetaxel (DC) and Nivolumab Variant #02 (CUI: 130690)
Cisplatin: 75 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4942
Docetaxel: 75 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4943
Nivolumab: 360 mg IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4944

In [15]:
neoadj_niv_eg_schedule = SystemicPhase.from_component_sigs(neoadj_w_nivolumab_eg.sigs)

In [16]:
neoadj_niv_eg_dates = neoadj_niv_eg_schedule.get_scheduled_dates(pd.to_datetime("2020-02-05"))

In [17]:
for date, component_parts in neoadj_niv_eg_dates.items():
    dosing = {component_part.component.name: component_part.get_dose()
              for component_part in component_parts}
    print(f"{date}: {dosing}")

2020-02-06 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Docetaxel': (75.0, 'mg/m^2'), 'Nivolumab': (360.0, 'mg')}
2020-02-27 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Docetaxel': (75.0, 'mg/m^2'), 'Nivolumab': (360.0, 'mg')}
2020-03-19 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Docetaxel': (75.0, 'mg/m^2'), 'Nivolumab': (360.0, 'mg')}


## Example with ongoing

In [18]:
with so.Session(engine) as session:
    ici_opt_chemo_sq = nsclc_trt_generator.get_protocol_from_phase_union("ici_monotherapy_or_ici_with_chemotherapy_squamous", session)

In [19]:
ici_opt_chemo_sq[0].phases[0]

ici_monotherapy_or_ici_with_chemotherapy_squamous (Phase 1)
Atezolizumab monotherapy Variant #04 129738
Atezolizumab monotherapy Variant #06 129740
Cemiplimab monotherapy Variant #05 130560
Durvalumab monotherapy Variant #05 161285
Pembrolizumab monotherapy Variant #07 133542
Pembrolizumab monotherapy Variant #08 133543
Pembrolizumab monotherapy Variant #10 133545
Pembrolizumab monotherapy Variant #12 133547
Pembrolizumab monotherapy Variant #25 133560
Tislelizumab monotherapy Variant #01 134156
Carboplatin and Paclitaxel (CP) and Durvalumab Variant #03 144931
Carboplatin and Paclitaxel (CP) and Penpulimab Variant #01 141817
Carboplatin and nab-Paclitaxel (CnP) and Serplulimab Variant #01 141809
Carboplatin and nab-Paclitaxel (CnP) and Serplulimab Variant #02 141810
Carboplatin and Paclitaxel (CP) and Ipilimumab Variant #03 130411
Carboplatin and Paclitaxel (CP) and Camrelizumab Variant #01 130405
Carboplatin, Pemetrexed, Pembrolizumab Variant #01 130508
Cisplatin, Pemetrexed, Pembrolizumab Variant #01 131081
Carboplatin and Paclitaxel (CP) and Pembrolizumab Variant #04 134523
Carboplatin and nab-Paclitaxel (CnP) and Pembrolizumab Variant #02 134525
Carboplatin and Paclitaxel (CP) and Nivolumab Variant #01 130415
Carboplatin and Paclitaxel (CP) and Nivolumab Variant #02 130416
Carboplatin and Paclitaxel (CP) and Nivolumab Variant #03 130417
Carboplatin and Paclitaxel (CP) and Nivolumab Variant #04 130418
Carboplatin and Paclitaxel (CP) and Ipilimumab, Nivolumab Variant #01 130412
Carboplatin, Pemetrexed, Ipilimumab, Nivolumab Variant #01 130506
Carboplatin, Pemetrexed, Ipilimumab, Nivolumab Variant #02 130507
Cisplatin, Pemetrexed, Ipilimumab, Nivolumab Variant #01 131079
Carboplatin and Paclitaxel (CP) and Sugemalimab Variant #01 130431
Carboplatin, Pemetrexed, Sugemalimab Variant #01 130510
Carboplatin and nab-Paclitaxel (CnP) and Atezolizumab Variant #02 130491
Carboplatin and Paclitaxel (CP) and Pembrolizumab Variant #01 130420
Carboplatin and nab-Paclitaxel (CnP) and Pembrolizumab Variant #01 130496
Carboplatin and Gemcitabine (GCb) and Sintilimab Variant #01 144706
Cisplatin and Gemcitabine (GC) and Sintilimab Variant #01 144726
Carboplatin and Paclitaxel (CP) and Pembrolizumab Variant #06 161278
Carboplatin, Pemetrexed, Pembrolizumab Variant #06 161279
Cisplatin, Pemetrexed, Pembrolizumab Variant #07 161282
Carboplatin and nab-Paclitaxel (CnP) and Tislelizumab Variant #01 130332
Carboplatin and Paclitaxel (CP) and Tislelizumab Variant #01 130432

### ComponentPart-specific ongoing

Pemetrexed is ongoing whereas other components have fixed cycle number. My interpretation is that therefore, pemetrexed should only have the potential to continue while other components in the same CycleSet ("implicit portion") are ongoing. This is achieved by changing the CycleEvents such that Pemetrexed's ongoing flag can result in `cycle_events.check_active_this_cycle(cycle_number)` resulting in a true value. 

This is only possible `if (not self._discontinued) and ((cycle_num-max(self.every)) % self.ongoing_step == 0)`. I.e. two things have to be true: there have been no failures to ongoing after passing this if, and the magnitude of the ongoing step must be consistent with the difference between the last explicit cycle and the current cycle number.

Therefore, after passing the if check, if by psuedorandom chance that cycle is NOT used, there will be no more pemetrexed cycles. This is intended to reflect discontinuation of a component due to toxicity.


Note that the `cycle_events.check_active_this_cycle(cycle_number)` check is only reached if `cycle_set.get_one_systemic_cycle()` is ran. Therefore, pemetrexed (in this case) only gets the chance to pass the if check if the CycleSet ("implicit portion") is still active.

In [20]:
ici_opt_chemo_sq[0].phases[0].options[17]

Cisplatin, Pemetrexed, Pembrolizumab Variant #01 (CUI: 131081)
Cisplatin: 75 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3,4. None Timing relative to other components: third. SigID: 6338
Pembrolizumab: 200 mg IV over 30.0 minutes once on day(s) 1 in cycle(s) 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35. None Timing relative to other components: first. SigID: 6339
Pemetrexed: 500 mg/m^2 IV over 10.0 minutes once on day(s) 1 in cycle(s) 1,(+1). None Timing relative to other components: second. SigID: 6340

In [21]:
inc_pemetrexed_eg = SystemicPhase.from_component_sigs(ici_opt_chemo_sq[0].phases[0].options[17].sigs)

In [22]:
inc_pemetrexed_eg.reset()
inc_pemetrexed_eg_dates = inc_pemetrexed_eg.get_scheduled_dates(pd.to_datetime("2020-08-05"))

In [23]:
for date, component_parts in inc_pemetrexed_eg_dates.items():
    dosing = {component_part.component.name: component_part.get_dose()
              for component_part in component_parts}
    print(f"{date}: {dosing}")

2020-08-06 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2020-08-27 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2020-09-17 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2020-10-08 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2020-10-29 00:00:00: {'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2020-11-19 00:00:00: {'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2020-12-10 00:00:00: {'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2020-12-31 00:00:00: {'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2021-01-21 00:00:00: {'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2021-02-11 00:00:00: {'Pembrolizumab': (200.0, 'mg'), 'Pemetrexed': (500.0, 'mg/m^2')}
2021-0

### Whole CycleSet ("implicit portion") ongoing

For advanced treatments, such as monotherapies, it is common to have all component(s) ongoing. In this case, it is inappropriate to avoid ongoing due to their not being any ComponentParts with explicit cycles. Therefore, CycleSet needs to a) detect this situation b) facilitate ongoing in this case.

If all component parts having ongoing, the minimum ongoing magnitude will be taken as the cycle step size. Any failure to pass the psuedorandom ongoing check will be taken as CUDPUT failure -> therefore activating the discontinued flag (like with the ComponentPart specific implementation) for the CycleSet. 

Relevant:

'For regimens with no definite stopping point, report merely the cycle length, e.g., "4-week cycles". This is generally understood to be equivalent to "4-week cycles until progression, unacceptable side effects, or patient preference to discontinue."'

https://hemonc.org/wiki/Style_guide#Number_of_cycles_(including_singular_courses)

In [24]:
ici_opt_chemo_sq[0].phases[0].options[0]

Atezolizumab monotherapy Variant #04 (CUI: 129738)
Atezolizumab: 840 mg IV over 60.0 minutes once on day(s) 1 in cycle(s) 1,(+1). None SigID: 1082

In [25]:
atezo_scheduler = SystemicPhase.from_component_sigs(ici_opt_chemo_sq[0].phases[0].options[0].sigs)

In [26]:
atezo_scheduler

SystemicPhase(cycle_sets=[CycleSet(seed=None, cycle_length_lb=14, cycle_length_ub=14, cycle_len_unit='day', cycle_sigs='14-day cycles', trios=[Trio(per_cycle_days=PerCycleDays(every=[1], base=[1], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='Atezolizumab', cui=49, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b368d50>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=840.0, dosemaxnum=840.0, doseunit='mg', doseunit_cui=134444, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='primary systemic', class_field='iv intermittent canonical sig'), cycle_events=CycleEvents(seed=None, every=[1], base=[1], optional=[], ongoing_step=1, first_event=1), other_info=OtherSigInfo(in_parens=None, tail='-', temp=None, sig_id=1082, date_added=datetime.datetime(2023, 11, 17, 0, 0)))], l

Note that opposite to wishes any pathway going through cycle events has lower effective continue prob

because cycleset * component < cycleset,  when 0 < component probs < 1 for any cycleset prob

only soln is to untangle the paths

In [27]:
atezo_scheduler.reset()
print(atezo_scheduler.get_scheduled_dates(pd.to_datetime("2022-08-05")))
atezo_scheduler.reset()

defaultdict(<class 'list'>, {Timestamp('2022-08-06 00:00:00'): [ComponentPart(component=Component(name='Atezolizumab', cui=49, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b368d50>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=840.0, dosemaxnum=840.0, doseunit='mg', doseunit_cui=134444, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='primary systemic', class_field='iv intermittent canonical sig')], Timestamp('2022-08-20 00:00:00'): [ComponentPart(component=Component(name='Atezolizumab', cui=49, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b368d50>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=840.0, dosemaxnum=840.0, doseunit='mg', doseunit_cui=134444, divided=False, dose_cap=Dos

In [28]:
atezo_scheduler.cycle_sets[0]._discontinued

False

## Non-systemic

First starting off by exploring how well systemic phase is or isn't able to represent phases that are not excludively systemic.

In [29]:
nsclc_registry.protocols

{'port_adjuvant_osimertinib': ProtocolYaml(protocol_id='port_adjuvant_osimertinib', source_phase_ids=['surgery_only', 'port', 'adj_osimertinib'], optional_phases=None),
 'surgery_then_port': ProtocolYaml(protocol_id='surgery_then_port', source_phase_ids=['surgery_only', 'port'], optional_phases=None),
 'neoadjuvant_or_perioperative_ici_w_chemo': ProtocolYaml(protocol_id='neoadjuvant_or_perioperative_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}}),
 'adjuvant_cisplatin_osimertinib': ProtocolYaml(protocol_id='adjuvant_cisplatin_osimertinib', source_phase_ids=['adjuvant_cisplatin_based', 'adjuvant_osimertinib'], optional_phases=None),
 'conc_crt_followed_by_durvalumab': ProtocolYaml(protocol_id='conc_crt_followed_by_durvalumab', source_phase_ids=['conc_crt', 'consolidation_durvalumab'], optional_phases=None),
 'conc_crt_foll

In [30]:
with so.Session(engine) as session:
    conc_crt_durv_consol = nsclc_trt_generator.get_protocols_from_protocol_yaml(
        nsclc_registry.protocols["conc_crt_followed_by_durvalumab"], session
        )

In [31]:
conc_crt_durv_consol[0]

Treatment Protocol: conc_crt_followed_by_durvalumab (2 phase(s))

In [32]:
conc_crt_durv_consol[0].phases[0].options[1]

Carboplatin and Paclitaxel (CP) and RT Variant #06 (CUI: 130428)
Carboplatin: 2 AUC IV over 30.0 minutes once per day on day(s) [1,36,7] in cycle(s) 1. None SigID: 3696
External beam radiotherapy: 200 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3697
Paclitaxel: 45 mg/m^2 IV over 1.0 to 3.0 hours once per day on day(s) [1,36,7] in cycle(s) 1. None SigID: 3698

In [33]:
print(f"inparens: {conc_crt_durv_consol[0].phases[0].options[1].sigs[1].inparens}")
print(f"tail: {conc_crt_durv_consol[0].phases[0].options[1].sigs[1].tail}")

inparens: (total of 6000 cGy)
tail: fractions x 30 fractions


In [34]:
conc_cp_rt = conc_crt_durv_consol[0].phases[0].options[0]
conc_cp_rt

Carboplatin and Paclitaxel (CP) and RT Variant #05 (CUI: 130427)
Carboplatin: 2 AUC IV once per day on day(s) [1,36,7] in cycle(s) 1. None SigID: 3693
External beam radiotherapy: 6000 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3694
Paclitaxel: 40 mg/m^2 IV once per day on day(s) [1,36,7] in cycle(s) 1. None SigID: 3695

In [35]:
print(f"inparens: {conc_cp_rt.sigs[1].inparens}")
print(f"tail: {conc_cp_rt.sigs[1].tail}")

inparens: None
tail: -


In [36]:
conc_crt_durv_consol[0].phases[0].options[2]

Carboplatin and Paclitaxel (CP) and RT Variant #07 (CUI: 130429)
Carboplatin: 2 AUC IV over 30.0 minutes once per day on day(s) [1,36,7] in cycle(s) 1. None Timing relative to other components: second. SigID: 3699
External beam radiotherapy: 180 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3700
External beam radiotherapy: 200 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3701
Paclitaxel: 45 mg/m^2 IV over 1.0 to 3.0 hours once per day on day(s) [1,36,7] in cycle(s) 1. None Timing relative to other components: first. SigID: 3702

"Radiotherapy

    Concurrent radiation therapy 4,500 cGy in 25 daily fractions of 180 cGy each, administered five days per week, then 1,800 cGy in nine daily fractions of 200 cGy each, administered five days per week"

In [37]:
print(f"step number: {conc_crt_durv_consol[0].phases[0].options[2].sigs[1].step_number}")
print(f"inparens: {conc_crt_durv_consol[0].phases[0].options[2].sigs[1].inparens}")
print(f"tail: {conc_crt_durv_consol[0].phases[0].options[2].sigs[1].tail}")

step number: 1 of 2
inparens: None
tail: fractions x 25 fractions


In [38]:
print(f"step number: {conc_crt_durv_consol[0].phases[0].options[2].sigs[2].step_number}")
print(f"inparens: {conc_crt_durv_consol[0].phases[0].options[2].sigs[2].inparens}")
print(f"tail: {conc_crt_durv_consol[0].phases[0].options[2].sigs[2].tail}")

step number: 2 of 2
inparens: None
tail: fractions x 9 fractions


In [39]:
conc_crt_durv_consol[0].phases[0].options[3]

Carboplatin and Paclitaxel (CP) and RT Variant #08 (CUI: 130430)
Carboplatin: 2 AUC IV over 30.0 minutes once per day on day(s) [1,43,7] in cycle(s) 1. None Timing relative to other components: second. SigID: 3703
External beam radiotherapy: 6600 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3704
Paclitaxel: 50 mg/m^2 IV over 60.0 minutes once per day on day(s) [1,43,7] in cycle(s) 1. None Timing relative to other components: first. SigID: 3705

In [40]:
print(f"step number: {conc_crt_durv_consol[0].phases[0].options[3].sigs[1].step_number}")
print(f"inparens: {conc_crt_durv_consol[0].phases[0].options[3].sigs[1].inparens}")
print(f"tail: {conc_crt_durv_consol[0].phases[0].options[3].sigs[1].tail}")

step number: 1 of 1
inparens: None
tail: -


In [41]:
conc_crt_durv_consol[0].phases[0].options[4]

MVP and RT Variant #01 (CUI: 136988)
Cisplatin: 80 mg/m^2 IV once per day on day(s) [1,29,28] in cycle(s) 1. None SigID: 18980
External beam radiotherapy: 5600 cGy Not specified on day(s) 1,2,3,4,5,8,9,10,11,12,15,16,17,18,19,29,30,31,32,33,36,37,38,39,40,43,44,45,46 in cycle(s) 1. None Timing relative to other components: on. SigID: 18981
Mitomycin: 8 mg/m^2 IV once per day on day(s) [1,29,28] in cycle(s) 1. None SigID: 18982
Vindesine: 3 mg/m^2 IV once per day on day(s) 1,8,29,36 in cycle(s) 1. None SigID: 18983

In [42]:
print(f"step number: {conc_crt_durv_consol[0].phases[0].options[4].sigs[1].step_number}")
print(f"inparens: {conc_crt_durv_consol[0].phases[0].options[4].sigs[1].inparens}")
print(f"tail: {conc_crt_durv_consol[0].phases[0].options[4].sigs[1].tail}")
print(f"alldays: {conc_crt_durv_consol[0].phases[0].options[4].sigs[1].alldays}")

step number: 1 of 1
inparens: None
tail: -
alldays: 1,2,3,4,5,8,9,10,11,12,15,16,17,18,19,29,30,31,32,33,36,37,38,39,40,43,44,45,46


In [43]:
conc_crt_durv_consol[0].phases[0].options[4].sigs[1].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7fa42b42acf0>,
 'cycle_length_lb': 46.0,
 'dosecapunit_cui': None,
 'durationunit_cui': None,
 'regimen_cui': 31033,
 'targetleveltype': None,
 'id': 18981,
 'cycle_length_ub': '46',
 'dosemaxnum': '5600',
 'route': 'Not specified',
 'route_cui': None,
 'targetlevelunit': None,
 'branch': 'none',
 'cycle_length_unit': <Sigs_Cycle_length_unitEnum.DAY: 'day'>,
 'doseminnum': '5600',
 'frequency': None,
 'seqrelwhat': None,
 'targetlevelunit_cui': None,
 'alldays': '1,2,3,4,5,8,9,10,11,12,15,16,17,18,19,29,30,31,32,33,36,37,38,39,40,43,44,45,46',
 'branch_type': None,
 'cyclesigs': '46-day course',
 'doseunit': 'cGy',
 'frequency_cui': None,
 'sequence': 'on',
 'temp': None,
 'class_field': <Sigs_Class_fieldEnum.RAD_SIG: 'rad sig'>,
 'date_added': datetime.datetime(2024, 3, 17, 0, 0),
 'doseunit_cui': 139561.0,
 'inparens': None,
 'step_number': '1 of 1',
 'timing_sequence': '1',
 'component': 'External beam radiotherapy',
 '

In [44]:
conc_crt_durv_consol[0].phases[0].options[5]

MVP and RT Variant #02 (CUI: 141885)
Cisplatin: 80 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3,4. None SigID: 18984
External beam radiotherapy: 200 cGy Not specified on day(s) 2,3,4,5,6,9,10,11,12,13,16,17,18,19,20 in cycle(s) 1,3. None Timing relative to other components: on. SigID: 18985
Mitomycin: 8 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3,4. None SigID: 18986
Vindesine: 3 mg/m^2 IV once per day on day(s) [1,8,7] in cycle(s) 1,2,3,4. None SigID: 18987

In [45]:
print(f"step number: {conc_crt_durv_consol[0].phases[0].options[5].sigs[1].step_number}")
print(f"inparens: {conc_crt_durv_consol[0].phases[0].options[5].sigs[1].inparens}")
print(f"tail: {conc_crt_durv_consol[0].phases[0].options[5].sigs[1].tail}")
print(f"alldays: {conc_crt_durv_consol[0].phases[0].options[5].sigs[1].alldays}")

step number: 1 of 1
inparens: None
tail: -
alldays: 2,3,4,5,6,9,10,11,12,13,16,17,18,19,20


In [46]:
# is having all days here an indicator that the dose is daily?
# No - above also has alldays listed
# Maybe return to threshold + amount idea?
conc_crt_durv_consol[0].phases[0].options[5].sigs[1].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7fa42b428590>,
 'cycle_length_lb': 28.0,
 'dosecapunit_cui': None,
 'durationunit_cui': None,
 'regimen_cui': 31033,
 'targetleveltype': None,
 'id': 18985,
 'cycle_length_ub': '28',
 'dosemaxnum': '200',
 'route': 'Not specified',
 'route_cui': None,
 'targetlevelunit': None,
 'branch': 'none',
 'cycle_length_unit': <Sigs_Cycle_length_unitEnum.DAY: 'day'>,
 'doseminnum': '200',
 'frequency': None,
 'seqrelwhat': None,
 'targetlevelunit_cui': None,
 'alldays': '2,3,4,5,6,9,10,11,12,13,16,17,18,19,20',
 'branch_type': None,
 'cyclesigs': '28-day cycle for 2 cycles',
 'doseunit': 'cGy',
 'frequency_cui': None,
 'sequence': 'on',
 'temp': None,
 'class_field': <Sigs_Class_fieldEnum.RAD_SIG: 'rad sig'>,
 'date_added': datetime.datetime(2024, 6, 17, 0, 0),
 'doseunit_cui': 139561.0,
 'inparens': None,
 'step_number': '1 of 1',
 'timing_sequence': '1,3',
 'component': 'External beam radiotherapy',
 'divided': False,
 'durationma

Why is this a 6 week course with days happenning to align to once a week and not 6 week-long cycles? No idea!

In [47]:
conc_cp_rt.sigs

In [48]:
# trying to understand dosing intent
# below we see cycle (/"course") of len 6 unit week and timing 1, alldays None
# this would seem to imply no days in any cycle (/"course") is rt delivered
# but that doesn't seem right
conc_cp_rt.sigs[1].study

'WJTOG0105'

Clear information provided in hemonc section: https://hemonc.org/wiki/Non-small_cell_lung_cancer#Carboplatin_and_Paclitaxel_(CP)_and_RT

"Concurrent radiation therapy 6,000 cGy in 30 daily fractions of 200 cGy each, administered five days per week"

Copied description from variant 1 rather than variant 5 as variant 5 had dosing not consistent with sig details and course length not consistent with sig details, as well as not listing study code in table (unlike variant 1)

Note therefore that dose values in min & max are TOTAL and not daily fractions

IDEA: Is it suitable to have a threshold for "there's no way this is meant to be daily" for __ cGy? 

Here https://www.annalsthoracicsurgery.org/article/S0003-4975(02)04681-7/fulltext, 1500 Gy is delivered in a single fraction

Maybe a threshold would need to work w/ number of doses? E.g. 30 fractions of 1500 would not be likely to occur? How does this work with the assumption of weekday doses when day not specified? Can I assume that for a single dose the 1 day would be correctly picked up and it's the text above that threw off scraping in a way which wouldn't occur for one off?

Reading full article: Questionable to apply a threshold for too high to be a single dose in isolation of other information, they are planning 2500 in future.

IDEA: can in_parens be used to differentiate when dosing is given as total vs daily?

RESULT: this was inconsistent even if looking for in_parens OR tail

DECISION: 

SUSPECT_RT_DOSE_IS_TOTAL_THRESHOLD = 1000

if recorded dose >= SUSPECT_RT_DOSE_IS_TOTAL_THRESHOLD AND multiple deliveries, ASSUME recorded dose is total dose and derive daily dose. Try to be more confident with in parens first

Obtain delivery days via:
- if alldays alldays else infer

In [49]:
# Getting daily dose size:
# if dose presented as total
# Get days of delivery
# daily dose = total dose / days of delivery
# else dose per stated dose range

In [50]:
# This is the reason for the break below - no alldays string to split
conc_cp_rt.sigs[1].alldays

In [51]:
# breaks on get_continue(ho_timing) through PerCycleDays
# because sig.alldays is none
# NO LONGER ERRORS BUT NOT ROLLOUT READY - see empty lists for PerCycleDays
SystemicPhase.from_component_sigs(conc_cp_rt.sigs)

SystemicPhase(cycle_sets=[CycleSet(seed=None, cycle_length_lb=6, cycle_length_ub=6, cycle_len_unit='week', cycle_sigs='6-week course', trios=[Trio(per_cycle_days=PerCycleDays(every=[1, 8, 15, 22, 29, 36], base=[1, 8, 15, 22, 29, 36], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='Carboplatin', cui=88, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b36b110>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once per day', frequency_cui=139562, dose_info=StandardDoseInfo(seed=None, doseminnum=2.0, dosemaxnum=2.0, doseunit='AUC', doseunit_cui=134425, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='primary systemic', class_field='iv intermittent canonical sig'), cycle_events=CycleEvents(seed=None, every=[1], base=[1], optional=[], ongoing_step=None, first_event=1), other_info=OtherSigInfo(in_parens=None, tail='-', temp=None, sig_id=3693, date_added

In [52]:
import re

from pydantic import BaseModel

In [53]:
class ImpliedRTCycleDays(BaseModel):
    days_implied_fractions: int | None
    days_implied_cycle_length: int
    sig_tail: str | None

    @classmethod
    def from_sig(cls, sig, cycle_len, cycle_len_unit):
        # NOTE cycle len not from sig - this is cycle set's responsibility
        # when cycle length is non-constant this may matter
        # therefore both length and unit should be passed by cycle set
        return cls(days_implied_fractions = cls._try_get_fractions(sig.tail),
                   days_implied_cycle_length = cls._from_cycle_length(cycle_len, cycle_len_unit),
                   sig_tail = sig.tail)

    @classmethod
    def from_trio(cls, trio, cycle_len, cycle_len_unit):
        tail = trio.other_info.tail
        return cls(days_implied_fractions = cls._try_get_fractions(tail),
                   days_implied_cycle_length = cls._from_cycle_length(cycle_len, cycle_len_unit),
                   sig_tail = tail)

    def get_number_days(self) -> int:
        # ASSUMPTION: One fraction per delivery day
        # Fractions appears a more reliable indicator of intended delivery days
        # than cycle length. See e.g. variant cui 130429 Part 2 of 2 RT sig
        return self.days_implied_fractions or self.days_implied_cycle_length

    @staticmethod
    def _try_get_fractions(sig_tail):
        if sig_tail:
            match = re.search(r"(\d+)\s+(?=fractions)", sig_tail)
            return int(match.group(1)) if match else None

    @staticmethod
    def _from_cycle_length(cycle_len, cycle_len_unit):
        delivery_days = {"week": cycle_len * 5,
                         "day": cycle_len * 5/7,
                         "month": cycle_len * 20}
        return delivery_days[cycle_len_unit]

In [54]:
implied_rt_days = ImpliedRTCycleDays.from_sig(conc_crt_durv_consol[0].phases[0].options[2].sigs[2],
                            # BAD but done here for convenience
                            cycle_len=conc_crt_durv_consol[0].phases[0].options[2].sigs[2].cycle_length_lb,
                            cycle_len_unit=conc_crt_durv_consol[0].phases[0].options[2].sigs[2].cycle_length_unit)

In [55]:
implied_rt_days

ImpliedRTCycleDays(days_implied_fractions=9, days_implied_cycle_length=35, sig_tail='fractions x 9 fractions')

In [56]:
implied_rt_days.get_number_days()

9

In [57]:
class InparensRTDosing(BaseModel):
    total_dose_inparens_value: float | None
    total_dose_inparens_unit: str | None

    @classmethod
    def from_sig(cls, sig):
        if sig.inparens:
            cls.from_inparens(sig.inparens)

    @classmethod
    def from_trio(cls, trio):
        if trio.other_info.in_parens:
            cls.from_inparens(trio.other_info.in_parens)

    @classmethod
    def from_inparens(cls, inparens):
        # Two capturing groups: 1+ digits (\d+); 1+ letters ([a-zA-Z]+)
        match = re.search(r"total\sof\s(\d+)\s([a-zA-Z]+)", inparens)
        value = float(match.group(1)) if match else None
        unit = match.group(2) if match else None
        return cls(total_dose_inparens_value = value,
                    total_dose_inparens_unit = unit)

In [58]:
InparensRTDosing.from_sig(conc_crt_durv_consol[0].phases[0].options[1].sigs[1])

---

In [59]:
# 10.1200/JCO.2000.18.19.3400 <-- why is this listed as for nsclc condition 
# when ref has it as for sclc? Condition for is nsclc in csv not just db
conc_crt_durv_consol[0].phases[0].options[-1]

Teniposide and WBRT Variant #01 (CUI: 134104)
External beam radiotherapy: 3000 cGy Not specified on day(s) None in cycle(s) 1. None within 3 weeks of initiation Teniposide. SigID: 22554
Teniposide: 120 mg/m^2 IV over 60.0 minutes once per day on day(s) [1,5,2] in cycle(s) 1,2,3,4,5,6,7,8,9,10,11,12. None SigID: 22555

In [60]:
conc_crt_durv_consol[0].phases[0].options[2]

Carboplatin and Paclitaxel (CP) and RT Variant #07 (CUI: 130429)
Carboplatin: 2 AUC IV over 30.0 minutes once per day on day(s) [1,36,7] in cycle(s) 1. None Timing relative to other components: second. SigID: 3699
External beam radiotherapy: 180 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3700
External beam radiotherapy: 200 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3701
Paclitaxel: 45 mg/m^2 IV over 1.0 to 3.0 hours once per day on day(s) [1,36,7] in cycle(s) 1. None Timing relative to other components: first. SigID: 3702

In [61]:
# HOW TO DEAL WITH NONE DAYS MULTIPLE DEPENDENT SIGS PROBLEM
# 1. Identify NONE alldays sigs
# 2. Identify any relative sequencing days expectations
# 3. Populate each based on delivery days, sequentially
# 4. Success!

In [62]:
# ^ but populate WHAT
# DECISION: Do not need to maintain PerCycleDays as faithful rep of sig alldays
# - it's not like I'm destroying the sig representation
# PerCycleDays is for which days to roll out - sig all days is just a means to that end

In [63]:
def get_5_ints_2_gap_sequence(total_len: int, start_num=1) -> list[int]:
    """
    Returns a sequence like 1, 2, 3, 4, 5, 8, 9, ... 

    Integers in 6+7x, 7+7x (where x zero or int) never appear regardless of start. 
    
    E.g. 
    total_len=7, start_num=1 => [1, 2, 3, 4, 5, 8, 9]
    total_len=7, start_num=6 => [8, 9, 10, 11, 12, 15, 16]
    """
    return [(i + 1 + 2*(i//5)) for i in range(total_len+start_num)][start_num-1:-1]

In [64]:
get_5_ints_2_gap_sequence(15)

[1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 15, 16, 17, 18, 19]

In [65]:
get_5_ints_2_gap_sequence(15, start_num=6)

[8, 9, 10, 11, 12, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26]

----

In [66]:
InparensRTDosing.from_sig(conc_crt_durv_consol[0].phases[0].options[1].sigs[1])

In [67]:
ImpliedRTCycleDays.from_sig(conc_crt_durv_consol[0].phases[0].options[2].sigs[2],
                            # BAD but done here for convenience
                            cycle_len=conc_crt_durv_consol[0].phases[0].options[2].sigs[2].cycle_length_lb,
                            cycle_len_unit=conc_crt_durv_consol[0].phases[0].options[2].sigs[2].cycle_length_unit)

ImpliedRTCycleDays(days_implied_fractions=9, days_implied_cycle_length=35, sig_tail='fractions x 9 fractions')

In [68]:
implied_rt_days = ImpliedRTCycleDays.from_sig(conc_crt_durv_consol[0].phases[0].options[2].sigs[2],
                            # BAD but done here for convenience
                            cycle_len=conc_crt_durv_consol[0].phases[0].options[2].sigs[2].cycle_length_lb,
                            cycle_len_unit=conc_crt_durv_consol[0].phases[0].options[2].sigs[2].cycle_length_unit)

In [69]:
# Need way to get start int - in this case based on prev part
get_5_ints_2_gap_sequence(implied_rt_days.get_number_days())

[1, 2, 3, 4, 5, 8, 9, 10, 11]

In [70]:
conc_crt_durv_consol[0].phases[0].options[2].sigs[2].cyclesigs

'7-week course'

'Although there are various terms used to describe a cycle of treatment (e.g. round or course are often used in the literature) we strongly encourage the consistent use of the word cycle for regimens with two or more planned repetitions

If there is only one course of treatment, as in a course of chemoradiotherapy or of induction, use "One course". '

https://hemonc.org/wiki/Style_guide#Number_of_cycles_(including_singular_courses)

In [71]:
# define chemoradiotherapy course? set these rules for that?

In [72]:
conc_crt_durv_consol[0].phases[0].options[2]

Carboplatin and Paclitaxel (CP) and RT Variant #07 (CUI: 130429)
Carboplatin: 2 AUC IV over 30.0 minutes once per day on day(s) [1,36,7] in cycle(s) 1. None Timing relative to other components: second. SigID: 3699
External beam radiotherapy: 180 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3700
External beam radiotherapy: 200 cGy Not specified on day(s) None in cycle(s) 1. None SigID: 3701
Paclitaxel: 45 mg/m^2 IV over 1.0 to 3.0 hours once per day on day(s) [1,36,7] in cycle(s) 1. None Timing relative to other components: first. SigID: 3702

In [73]:
rt_parts_eg_scheduler = SystemicPhase.from_component_sigs(conc_crt_durv_consol[0].phases[0].options[2].sigs)
rt_parts_eg_scheduler

SystemicPhase(cycle_sets=[CycleSet(seed=None, cycle_length_lb=7, cycle_length_ub=7, cycle_len_unit='week', cycle_sigs='7-week course', trios=[Trio(per_cycle_days=PerCycleDays(every=[1, 8, 15, 22, 29, 36], base=[1, 8, 15, 22, 29, 36], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='Carboplatin', cui=88, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7fa42b36b110>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once per day', frequency_cui=139562, dose_info=StandardDoseInfo(seed=None, doseminnum=2.0, dosemaxnum=2.0, doseunit='AUC', doseunit_cui=134425, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='primary systemic', class_field='iv intermittent canonical sig'), cycle_events=CycleEvents(seed=None, every=[1], base=[1], optional=[], ongoing_step=None, first_event=1), other_info=OtherSigInfo(in_parens=None, tail='-', temp=None, sig_id=3699, date_added

In [74]:
rt_trios = [trio for trio in rt_parts_eg_scheduler.cycle_sets[0].trios if trio.component_part.class_field=="rad sig"]

In [75]:
rt_trios.sort(key=lambda x: x.component_part.step_int)
rt_trios

[Trio(per_cycle_days=PerCycleDays(every=[], base=[], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='External beam radiotherapy', cui=1608, drug_object=None), step_int=1, step_max=2, route='Not specified', route_cui=None, frequency=None, frequency_cui=None, dose_info=StandardDoseInfo(seed=None, doseminnum=180.0, dosemaxnum=180.0, doseunit='cGy', doseunit_cui=139561, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='locoregional', class_field='rad sig'), cycle_events=CycleEvents(seed=None, every=[1], base=[1], optional=[], ongoing_step=None, first_event=1), other_info=OtherSigInfo(in_parens=None, tail='fractions x 25 fractions', temp=None, sig_id=3700, date_added=datetime.datetime(2023, 11, 17, 0, 0))),
 Trio(per_cycle_days=PerCycleDays(every=[], base=[], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='External beam radiotherapy', cui=1608, drug_o

In [76]:
def has_no_cycle_days(trio):
    return not any(trio.per_cycle_days.__dict__.values())

has_no_cycle_days(rt_trios[0])

True

In [79]:
from collections import defaultdict

In [85]:
within_cycle_administration = defaultdict(list)
eg_trio = rt_trios[0]
cycle_start_date = pd.to_datetime("2018-01-01")
cycle_len = rt_parts_eg_scheduler.cycle_sets[0].get_cycle_length()
cycle_len_unit = rt_parts_eg_scheduler.cycle_sets[0].cycle_len_unit
start_day = 1

for trio in rt_trios:
    if has_no_cycle_days(trio):
        num_days = ImpliedRTCycleDays.from_trio(trio, cycle_len, cycle_len_unit).get_number_days()
        new_days = get_5_ints_2_gap_sequence(num_days, start_day)
        dates = [cycle_start_date + pd.DateOffset(days=day) for day in new_days]
        for date in dates:
            within_cycle_administration[date].append(trio.component_part)
        start_day = new_days[-1] + 1

In [87]:
within_cycle_administration

defaultdict(list,
            {Timestamp('2018-01-02 00:00:00'): [ComponentPart(component=Component(name='External beam radiotherapy', cui=1608, drug_object=None), step_int=1, step_max=2, route='Not specified', route_cui=None, frequency=None, frequency_cui=None, dose_info=StandardDoseInfo(seed=None, doseminnum=180.0, dosemaxnum=180.0, doseunit='cGy', doseunit_cui=139561, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='locoregional', class_field='rad sig')],
             Timestamp('2018-01-03 00:00:00'): [ComponentPart(component=Component(name='External beam radiotherapy', cui=1608, drug_object=None), step_int=1, step_max=2, route='Not specified', route_cui=None, frequency=None, frequency_cui=None, dose_info=StandardDoseInfo(seed=None, doseminnum=180.0, dosemaxnum=180.0, doseunit='cGy', doseunit_cui=139561, divided=False, dose_cap=DoseCap(dosecapnum=None, dosecapunit=None, dosecapunit_cui=None)), duration=None, role='locore

In [78]:
new_days

[1,
 2,
 3,
 4,
 5,
 8,
 9,
 10,
 11,
 12,
 15,
 16,
 17,
 18,
 19,
 22,
 23,
 24,
 25,
 26,
 29,
 30,
 31,
 32,
 33]